# 🗂️ Manage Data — Join CSV Files

This notebook **concatenates two SAP log CSV files** into a single `output/logs.csv`  
that the FastAPI container reads on startup.

### Steps
1. Load both source CSVs and preview them  
2. Concatenate (vertical stack — same 44-column schema)  
3. Deduplicate on `_id` (UUID)  
4. Sort by `@timestamp`  
5. Summary of the combined dataset  
6. Save to `output/logs.csv`  

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Libraries loaded ✓')

Libraries loaded ✓


## 1 · Define Paths

In [2]:
# ── Paths (relative to repo root) ──────────────────────────────────────────
OUTPUT_DIR = '../output'

FILE_A = os.path.join(OUTPUT_DIR, '28_30__3.csv')   # first batch
FILE_B = os.path.join(OUTPUT_DIR, 'logs.csv')   # second batch
OUT    = os.path.join(OUTPUT_DIR, 'logs_joins.csv')           # combined → API reads this

for f in [FILE_A, FILE_B]:
    size_mb = os.path.getsize(f) / 1_048_576
    print(f'  {os.path.basename(f):30s}  {size_mb:7.1f} MB')

print(f'\nOutput → {OUT}')

  28_30__3.csv                       64.1 MB
  logs.csv                          318.9 MB

Output → ../output/logs_joins.csv


## 2 · Load & Preview Each File

In [3]:
print('Loading file A …')
df_a = pd.read_csv(FILE_A, dtype=str, keep_default_na=False, low_memory=False)
print(f'  File A : {len(df_a):>10,} rows  ×  {df_a.shape[1]} columns')

print('Loading file B …')
df_b = pd.read_csv(FILE_B, dtype=str, keep_default_na=False, low_memory=False)
print(f'  File B : {len(df_b):>10,} rows  ×  {df_b.shape[1]} columns')

print(f'\nTotal rows before dedup : {len(df_a) + len(df_b):,}')

Loading file A …
  File A :    120,600 rows  ×  44 columns
Loading file B …
  File B :    600,400 rows  ×  44 columns

Total rows before dedup : 721,000


In [4]:
# ── Quick sanity check: same columns? ──────────────────────────────────────
cols_a = set(df_a.columns)
cols_b = set(df_b.columns)

only_in_a = cols_a - cols_b
only_in_b = cols_b - cols_a

if not only_in_a and not only_in_b:
    print('✅  Both files share the same columns — safe to concatenate.')
else:
    if only_in_a:
        print(f'⚠️  Columns only in A : {only_in_a}')
    if only_in_b:
        print(f'⚠️  Columns only in B : {only_in_b}')
    print('   Missing columns will be filled with empty strings after concat.')

✅  Both files share the same columns — safe to concatenate.


In [5]:
# ── Date ranges per file ────────────────────────────────────────────────────
for label, df_tmp in [('File A', df_a), ('File B', df_b)]:
    ts = pd.to_datetime(df_tmp['@timestamp'], utc=True, errors='coerce')
    print(f'{label}  →  {ts.min()}  …  {ts.max()}')

File A  →  2026-03-28 00:00:00+00:00  …  2026-03-30 00:00:00+00:00
File B  →  2026-03-30 00:00:00+00:00  …  2026-04-04 00:00:00+00:00


## 3 · Concatenate

In [6]:
# Vertical stack — same schema, just more rows
df = pd.concat([df_a, df_b], ignore_index=True)

print(f'After concat  : {len(df):,} rows')

After concat  : 721,000 rows


## 4 · Deduplicate on `_id`

In [7]:
before = len(df)
df = df.drop_duplicates(subset='_id', keep='first')
dupes_removed = before - len(df)

print(f'Duplicates removed : {dupes_removed:,}')
print(f'After dedup        : {len(df):,} rows')

Duplicates removed : 0
After dedup        : 721,000 rows


## 5 · Sort by `@timestamp`

In [8]:
df['_ts_sort'] = pd.to_datetime(df['@timestamp'], utc=True, errors='coerce')
df = df.sort_values('_ts_sort').drop(columns='_ts_sort').reset_index(drop=True)

ts = pd.to_datetime(df['@timestamp'], utc=True, errors='coerce')
print(f'Date range : {ts.min()}  →  {ts.max()}')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

Date range : 2026-03-28 00:00:00+00:00  →  2026-04-04 00:00:00+00:00
Final shape: 721,000 rows × 44 columns


## 6 · Summary of Combined Dataset

In [9]:
log_cat = df['sap_function_log_type'].apply(
    lambda x: 'LLM' if str(x).startswith('LLM') else 'System'
)

print('=== Combined Dataset Summary ===')
print(f'  Total rows      : {len(df):,}')
print(f'  System log rows : {(log_cat == "System").sum():,}  '
      f'({(log_cat == "System").mean()*100:.1f}%)')
print(f'  LLM log rows    : {(log_cat == "LLM").sum():,}  '
      f'({(log_cat == "LLM").mean()*100:.1f}%)')
print(f'  Unique _id      : {df["_id"].nunique():,}')
print(f'  Columns         : {df.shape[1]}')

print('\n=== Log Type Breakdown ===')
print(df['sap_function_log_type'].value_counts().to_string())

print('\n=== Source Files ===')
print(f'  {os.path.basename(FILE_A)}')
print(f'  {os.path.basename(FILE_B)}')

=== Combined Dataset Summary ===
  Total rows      : 721,000
  System log rows : 432,600  (60.0%)
  LLM log rows    : 288,400  (40.0%)
  Unique _id      : 721,000
  Columns         : 44

=== Log Type Breakdown ===
sap_function_log_type
LLM_REQUEST    202049
INFO           159660
WARNING         79841
ERROR           60575
LLM_ERROR       57441
AUDIT           40094
DEBUG           39760
PERF            37063
LLM_TIMEOUT     28910
SECURITY        15607

=== Source Files ===
  28_30__3.csv
  logs.csv


## 7 · Save to `output/logs.csv`

> This is the file the FastAPI container reads on startup (`CSV_PATH = output/logs.csv`).  
> After saving, restart the container to pick up the new data.

In [10]:
df.to_csv(OUT, index=False)

size_mb = os.path.getsize(OUT) / 1_048_576
print(f'✅  Saved → {OUT}')
print(f'   {len(df):,} rows  |  {size_mb:.1f} MB')
print()
print('Next step: restart the container to reload the data.')
print('  podman-compose -f ../podman-compose.yml restart api')

✅  Saved → ../output/logs_joins.csv
   721,000 rows  |  382.3 MB

Next step: restart the container to reload the data.
  podman-compose -f ../podman-compose.yml restart api
